In [1]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
import xgboost as xgb
import yaml
import os
import logging
from contextlib import nullcontext
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

mlflow.set_tracking_uri("http://localhost:5000")

In [2]:
data_path = "../data/processed/data_scientists_features.csv"
data = pd.read_csv(data_path)

In [3]:
data.head()

,price,sqft,bedrooms,location,year_built,condition,house_age,price_per_sqft,bed_bath_ratio
0,495000,1527,2,Suburb,1956,Good,70,324.165029,1.333333
1,752000,2526,3,Downtown,1998,Excellent,28,297.703880,1.200000
2,319000,1622,2,Rural,1975,Fair,51,196.670777,1.333333
3,1210000,3102,4,Waterfront,2005,Excellent,21,390.070922,1.333333
4,462000,1835,2,Urban,1982,Good,44,251.771117,1.000000


In [6]:
encoder_location = OneHotEncoder(handle_unknown='ignore', sparse=False)

In [9]:
encoder_location.fit_transform(data[['location']])

d:\Project_Git\PredictHouse\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


array([[0., 0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1., 0.],
       [0.

In [ ]:
# load dataset
data_path = "../data/processed/data_scientists_features.csv"
data = pd.read_csv(data_path)
X = data.drop(columns=["price"],axis=1) # axes=1 for column
y = data["price"]



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [5]:
data.location.unique()

array(['Suburb', 'Downtown', 'Rural', 'Waterfront', 'Urban', 'Mountain'],
      dtype=object)

In [6]:
data.condition.unique()

array(['Good', 'Excellent', 'Fair', 'Poor'], dtype=object)

In [4]:
X_train.head()

,sqft,bedrooms,location,year_built,condition,house_age,price_per_sqft,bed_bath_ratio
76,2620,3,Downtown,1994,Excellent,32,278.625954,1.200000
42,1680,2,Rural,1959,Fair,67,185.714286,1.333333
49,2050,3,Urban,1990,Good,36,253.658537,1.500000
11,2230,3,Downtown,1986,Good,40,275.784753,1.500000
30,1680,2,Suburb,1968,Fair,58,236.904762,1.000000


In [3]:
from sklearn.feature_selection import RFE
from xgboost import XGBRegressor

# Use XGBoost for RFE to stay consistent
xgb_model = XGBRegressor(objective='reg:squarederror')
xgb_model.fit(X_train, y_train)

# RFE
rfe_selector = RFE(estimator=xgb_model, n_features_to_select=10)
rfe_selector.fit(X_train, y_train)
rfe_selected_features = X.columns[rfe_selector.support_]
rfe_ignored_features = X.columns[~rfe_selector.support_]

print("✅ Top 10 Selected Features by RFE:")
for feature in rfe_selected_features:
    print(f" - {feature}")

print("\n❌ Features Ignored by RFE:")
for feature in rfe_ignored_features:
    print(f" - {feature}")

# Store for config
selected_features_dict = {
    'rfe': list(rfe_selected_features)
}

# Filter datasets to use only selected features for experimentation
X_train = X_train[rfe_selected_features]
X_test = X_test[rfe_selected_features]

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, The experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:location: object, condition: object